<a href="https://colab.research.google.com/github/sophiegsmith/sentiment-product-recommendation/blob/main/01_data_cleaning_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets

In [ ]:
!pip install -q pandas pyarrow

# Loading the Datasets

In [ ]:
#load the libraries used
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [ ]:
#load the review data
reviews_url = "https://huggingface.co/datasets/debolut/amazon-reviews-2023-all-beauty-sample/resolve/main/reviews_sample.parquet"

reviews = pd.read_parquet(reviews_url)

print("Reviews shape:", reviews.shape)
reviews.head()

Reviews shape: (12219, 8)


,user_id,item_id,rating,title,text,helpful_vote,verified_purchase,timestamp
0,AHCXZZ6R4US43PBVXXVN47OTTVHQ,B0002KHTCK,5.0,Best Toothbrush Available,I have been using the Braun Oral B 3D toothbru...,22,False,1103814114
1,AHUSGOTGVU5LFFPGCWLOTZ33RH7A,B0002KHTCK,5.0,Should have gotten this earlier,Just used this thing this morning and oh boy.....,7,False,1103819630
2,AHUSGOTGVU5LFFPGCWLOTZ33RH7A,B0002KHTCK,5.0,Should have gotten this earlier,Just used this thing this morning and oh boy.....,7,False,1103819630
3,AHUSGOTGVU5LFFPGCWLOTZ33RH7A,B0002KHTCK,5.0,Should have gotten this earlier,Just used this thing this morning and oh boy.....,7,False,1103819630
4,AFL2ZDLKD6WRY5AG35RAWWKQTJFA,B000PKKAGO,4.0,Nice Shave...but beter than a razor?,At various times in my life I have rotated bet...,0,False,1197136923


In [ ]:
#load the product metadata
metadata_url = "https://huggingface.co/datasets/debolut/amazon-reviews-2023-all-beauty-sample/resolve/main/meta_sample.parquet"
metadata = pd.read_parquet(metadata_url)

print("Metadata shape:", metadata.shape)
metadata.head()

Metadata shape: (2216, 16)


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,item_id,bought_together,subtitle,author
0,All Beauty,"Zydeco Chop Chop Cajun Seasoning Base, 8 Ounce...",4.7,21,"[""All Natural blend of Dehydrated Onion, Dehyd...","[""Zydeco Chop Chop is a blend of Dehydrated On...",None,"{""hi_res"": ""['https://m.media-amazon.com/image...","{""title"": ""[]"", ""url"": ""[]"", ""user_id"": ""[]""}",BORELTH,[],"{""Package Dimensions"": ""9.61 x 7.17 x 3.07 inc...",B0BTLTVR1X,None,None,None
1,All Beauty,Sharonelle Natural Cream Soft Wax for Sensitiv...,4.0,1,[],[],50.0,"{""hi_res"": ""['https://m.media-amazon.com/image...","{""title"": ""[]"", ""url"": ""[]"", ""user_id"": ""[]""}",Sharonelle,[],"{""Item Form"": ""Wax"", ""Skin Type"": ""Sensitive"",...",B071G4NNH7,None,None,None
2,All Beauty,Organic Sweet Almond Oil and Fractionated Coco...,4.5,167,[],[],None,"{""hi_res"": ""['https://m.media-amazon.com/image...","{""title"": ""['Organic Sweet Almond Oil and Coco...",Shiny Leaf,[],"{""Brand"": ""Shiny Leaf"", ""Scent"": ""Almond Oil a...",B08LYT4Q2X,None,None,None
3,All Beauty,"Empty Brown Glass Spray Bottles2-Pack, Refilla...",3.7,53,[],[],None,"{""hi_res"": ""['https://m.media-amazon.com/image...","{""title"": ""['Good Quality for the Price' 'Nice...",Cherry,[],"{""Brand"": ""Cherry"", ""Material"": ""Glass"", ""Capa...",B089CSF11Y,None,None,None
4,All Beauty,"OSRSR Cryo Sticks,Unbreakable Stainless Steel ...",4.5,107,"[""PREMIUM QUALITY ICE GLOBES: OSRSR cryo stick...",[],17.49,"{""hi_res"": ""['https://m.media-amazon.com/image...","{""title"": ""['Cryo Sticks Ice Globes Facial Ski...",OSRSR,[],"{""Material"": ""Stainless Steel"", ""Brand"": ""OSRS...",B0924GT395,None,None,None


In [ ]:
#check the columns
print("REVIEW COLUMNS:")
print(reviews.columns.tolist())

print("\nMETADATA COLUMNS:")
print(metadata.columns.tolist())

REVIEW COLUMNS:
['user_id', 'item_id', 'rating', 'title', 'text', 'helpful_vote', 'verified_purchase', 'timestamp']

METADATA COLUMNS:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'item_id', 'bought_together', 'subtitle', 'author']


In [ ]:
#lets check how much data we are working with
print(f"Number of reviews: {len(reviews):,}")
print(f"Number of products: {len(metadata):,}")
print(f"Unique users: {reviews['user_id'].nunique():,}")
print(f"Unique products in reviews: {reviews['item_id'].nunique():,}")

Number of reviews: 12,219
Number of products: 2,216
Unique users: 2,087
Unique products in reviews: 2,216


# Inspecting Data

In [ ]:
# display a few of the complete review records
pd.set_option("display.max_colwidth", 100)

reviews[["item_id", "rating", "title", "text", "helpful_vote", "verified_purchase"]].head(10)

,item_id,rating,title,text,helpful_vote,verified_purchase
0,B0002KHTCK,5.0,Best Toothbrush Available,"I have been using the Braun Oral B 3D toothbrush for some time now, and although that unit does ...",22,False
1,B0002KHTCK,5.0,Should have gotten this earlier,Just used this thing this morning and oh boy...feel like I just get my teeth cleaned by the dent...,7,False
2,B0002KHTCK,5.0,Should have gotten this earlier,Just used this thing this morning and oh boy...feel like I just get my teeth cleaned by the dent...,7,False
3,B0002KHTCK,5.0,Should have gotten this earlier,Just used this thing this morning and oh boy...feel like I just get my teeth cleaned by the dent...,7,False
4,B000PKKAGO,4.0,Nice Shave...but beter than a razor?,At various times in my life I have rotated between using a razor vs a cordless shaver. While I p...,0,False
5,B000PKKAGO,5.0,Made me a Convert (from Razor Blades),I was skeptical about this shaver because of my past experience with electric razors. I want a v...,1,False
6,B000PKNPES,3.0,"Norelco is great, this one is just OK",I have used Norelco shavers for many years and prefer them to all other models I have tried. Th...,19,False
7,B000PKMA2G,3.0,Just OK,I have used Norelco shavers for many years and prefer them to all other models I have tried. Thi...,9,False
8,B000PKKAGO,3.0,Just OK,I have used Norelco shavers for many years and prefer them to all other models I have tried. Thi...,2,False
9,B000PKKAGO,4.0,Good for the frequent shaver,The Philips Norelco Arcitec shaving system is designed to look good and feel good in your hand. ...,5,False


In [ ]:
#disply a few product records
metadata[["item_id", "title", "average_rating", "rating_number", "price", "store"]].head(10)

,item_id,title,average_rating,rating_number,price,store
0,B0BTLTVR1X,"Zydeco Chop Chop Cajun Seasoning Base, 8 Ounce Resealable Bags (Pack of Two - 1 Pound Total)",4.7,21,None,BORELTH
1,B071G4NNH7,Sharonelle Natural Cream Soft Wax for Sensitive Skin in 14 oz. - 4 cans,4.0,1,50.0,Sharonelle
2,B08LYT4Q2X,"Organic Sweet Almond Oil and Fractionated Coconut Oil Bundle for Hair and Skin, 100% Pure and Na...",4.5,167,None,Shiny Leaf
3,B089CSF11Y,"Empty Brown Glass Spray Bottles2-Pack, Refillable 16 oz Containers for Essential Oils, Cleaning ...",3.7,53,None,Cherry
4,B0924GT395,"OSRSR Cryo Sticks,Unbreakable Stainless Steel Beauty Ice Facial Massage Tools for Woman,Skin Cas...",4.5,107,17.49,OSRSR
5,B081ZN3TD5,JPNK 4PCS Anti-Static Detangling Fine & Wide Tooth Shower Comb Set,4.1,58,None,JPNK
6,B07Q4YCBWD,Best Mom Ever French Milled Luxury Soap Set of 3 Bars (6.35oz),4.2,18,None,Jolitee
7,B0871ZTZSW,20 Sheets Nail Polish Stickers Glitter Gradient Nail Wraps Pure Color Shine Nail Tip Stickers Fu...,3.7,113,None,Blulu
8,B07CTQQLLJ,Staleks Pro Professional Cuticle Scissors Curved Manicure Pedicure Care Tools Nail Eyebrow Beard...,4.5,318,None,STALEKS
9,B08WHBSK6R,Kidskin- Foaming Body Wash with Tea Tree Oil- Helps Combat Body and Foot Odor and Stubborn Body ...,4.0,49,None,Kidskin


In [ ]:
#check the rating distribution
reviews["rating"].value_counts().sort_index()

#This is important because we need to see whether the sample has enough 1–2 star reviews to make negative sentiment meaningful
reviews["rating"].value_counts(normalize=True).sort_index() * 100

,proportion
rating,
1.0,5.491448
2.0,4.263851
3.0,10.385465
4.0,22.555037
5.0,57.304198


In [ ]:
#check review coverage per product
review_counts = (
    reviews.groupby("item_id")
    .size()
    .sort_values(ascending=False)
)

review_counts.describe()

,0
count,2216.000000
mean,5.513989
std,3.220390
min,3.000000
25%,3.000000
50%,4.000000
75%,7.000000
max,48.000000


In [ ]:
#This tells us whether we have a handful of heavily reviewed products and lots of products with only one review
review_counts.head(20)

,0
item_id,
B0B5XFVSXY,48
B0914JBBTV,27
B08HGZXLP6,23
B0BR4GL7Q6,22
B0BTJ6SYKB,21
B08YJV9YL2,21
B0BVQQ58G8,21
B08BZ1RHPS,20
B00KCTER3U,20


# Identifying products with enough reviews to make the analysis meaningful

In [ ]:
#examine review-count coverage
review_counts = reviews.groupby("item_id").size()

print("Products with at least 3 reviews: ", (review_counts >= 3).sum())
print("Products with at least 5 reviews: ", (review_counts >= 5).sum())
print("Products with at least 10 reviews: ", (review_counts >= 10).sum())
print("Products with at least 15 reviews: ", (review_counts >= 15).sum())
print("Products with at least 20 reviews: ", (review_counts >= 20).sum())

Products with at least 3 reviews:  2216
Products with at least 5 reviews:  1048
Products with at least 10 reviews:  266
Products with at least 15 reviews:  31
Products with at least 20 reviews:  10


In [ ]:
#run
review_counts.sort_values(ascending=False).head(20)

,0
item_id,
B0B5XFVSXY,48
B0914JBBTV,27
B08HGZXLP6,23
B0BR4GL7Q6,22
B0BTJ6SYKB,21
B08YJV9YL2,21
B0BVQQ58G8,21
B08BZ1RHPS,20
B00KCTER3U,20


In [ ]:
#create a product-level summary to see which products actually have enough reviews and what the products are
product_summary = (
    reviews.groupby("item_id")
    .agg(
        review_count=("rating", "count"),
        avg_review_rating=("rating", "mean"),
        avg_helpful_votes=("helpful_vote", "mean")
    )
    .sort_values("review_count", ascending=False)
)

product_summary.head(20)

,review_count,avg_review_rating,avg_helpful_votes
item_id,,,
B0B5XFVSXY,48,4.645833,0.562500
B0914JBBTV,27,4.370370,0.185185
B08HGZXLP6,23,4.869565,0.086957
B0BR4GL7Q6,22,4.136364,0.318182
B0BTJ6SYKB,21,4.523810,0.523810
B08YJV9YL2,21,4.190476,0.142857
B0BVQQ58G8,21,4.190476,0.095238
B08BZ1RHPS,20,4.300000,0.800000
B00KCTER3U,20,4.450000,0.300000


In [ ]:
# at least ten reviews minimum
product_summary[product_summary["review_count"] >= 10].head(20)

,review_count,avg_review_rating,avg_helpful_votes
item_id,,,
B0B5XFVSXY,48,4.645833,0.562500
B0914JBBTV,27,4.370370,0.185185
B08HGZXLP6,23,4.869565,0.086957
B0BR4GL7Q6,22,4.136364,0.318182
B0BTJ6SYKB,21,4.523810,0.523810
B08YJV9YL2,21,4.190476,0.142857
B0BVQQ58G8,21,4.190476,0.095238
B08BZ1RHPS,20,4.300000,0.800000
B00KCTER3U,20,4.450000,0.300000


In [ ]:
#connecting the products to its actual names
product_summary_named = (
    product_summary
    .reset_index()
    .merge(
        metadata[['item_id', 'title', 'average_rating', 'price', 'store']],
        on = 'item_id',
        how = 'left'
    )
)

product_summary_named[
    product_summary_named['review_count'] >= 10
].head(20)

,item_id,review_count,avg_review_rating,avg_helpful_votes,title,average_rating,price,store
0,B0B5XFVSXY,48,4.645833,0.562500,"MilyBest Fluffy Lashes False Eyelashes Natural Look Cat Eye Lashes, Dramatic Faux Mink Lashes 3D...",4.1,6.45,MilyBest
1,B0914JBBTV,27,4.370370,0.185185,"Bioré J-Beauty Makeup Removing Moisturizing Cleansing Jelly, Top Japanese Makeup Remover, Gentle...",4.4,11.95,Bioré
2,B08HGZXLP6,23,4.869565,0.086957,Empty Amber Glass Spray Bottles - (4 Pack) 16 oz with Labels Refillable Container for Essential ...,4.5,None,AYL
3,B0BR4GL7Q6,22,4.136364,0.318182,"Vegan Mia - USDA Organic Radiance Superfruit Cleansing Oil and Make-up Remover, Oil Cleanser and...",4.3,31.0,Vegan Mia
4,B0BTJ6SYKB,21,4.523810,0.523810,"Shakira Perfumes - Dance for Women - Long Lasting - Femenine, Charming and Modern Perfume - Frui...",4.4,29.34,Shakira
5,B08YJV9YL2,21,4.190476,0.142857,Bouquet Garni Body Shower White Musk - Unique Fragrance Decanting Engineering Method gives Long ...,4.0,13.99,Bouquet Garni
6,B0BVQQ58G8,21,4.190476,0.095238,"Beauty Bakerie Lip Whip Liquid Metallic Lipstick, Long Lasting Lip Color, Smudge Proof Makeup, B...",3.9,22.0,Beauty Bakerie
7,B08BZ1RHPS,20,4.300000,0.800000,"MD Complete Bright & Healthy Vitamin C+ Vitalizing Face Serum| with Vitamin C, Vitamin E and Her...",4.2,None,MD Complete by Dr. Brian Zelickson
8,B00KCTER3U,20,4.450000,0.300000,Retinol Face Moisturizer Cream Natural Facial moisturizing Cream 1.76 OZ with Ortho-Hydroxybenzo...,3.7,None,InstaNatural
9,B09G3KBK7Y,20,4.150000,1.300000,"Boiess Colognes Poppy mint For Moms, Babies & Kids | Natural Eau de Cologne | Clean & Fresh Scen...",3.7,22.9,Boiess


notes about this step above:
+ customer satisfaction --> most-reviewed products generally have average ratings above 4 stars
+ rating discrepancies --> wonder4 has 3.94 rating, while metadata has 3.4, look into these discrepancies, rather than an authoratative value
+ data quality --> the repeated-looking reviews from Step 6 still need to be checked for duplicates
+ business opportunity --> identify products with weaker customer feedback and search for better-reviewed alternatives in the same category

# Check and Clean Duplicates

In [ ]:
#check for exact duplicate rows
print("Exact duplicated rows: ", reviews.duplicated().sum())

#check duplicates based on product, rating, title, and review text
print(
    "Duplicate product-review combinations:",
    reviews.duplicated(
        subset=["item_id", "rating", "title", "text"]
    ).sum()
)

Exact duplicated rows:  2027
Duplicate product-review combinations: 2096


In [ ]:
#inspect repeated records:
duplicate_reviews = reviews[
    reviews.duplicated(
        subset=["item_id", "rating", "title", "text"],
        keep=False
    )
].sort_values(["item_id", "title"])

duplicate_reviews[
    ["item_id", "rating", "title", "text", "user_id"]
].head(15)

,item_id,rating,title,text,user_id
3403,B0000ZLF18,5.0,Soft comforting smell!,"Very nice smell! It has a soft smell and last a while, I have used it daily since my purchase!",AHC3VG2ALLSPHZDU2Z52YDHMLO4Q
3404,B0000ZLF18,5.0,Soft comforting smell!,"Very nice smell! It has a soft smell and last a while, I have used it daily since my purchase!",AHC3VG2ALLSPHZDU2Z52YDHMLO4Q
3405,B0000ZLF18,5.0,Soft comforting smell!,"Very nice smell! It has a soft smell and last a while, I have used it daily since my purchase!",AHC3VG2ALLSPHZDU2Z52YDHMLO4Q
3406,B0000ZLF18,5.0,Soft comforting smell!,"Very nice smell! It has a soft smell and last a while, I have used it daily since my purchase!",AHC3VG2ALLSPHZDU2Z52YDHMLO4Q
141,B00011JN5G,5.0,I love it,Great,AGEVTM6XYVZX454K6DTUH64LRROQ
142,B00011JN5G,5.0,I love it,Great,AGEVTM6XYVZX454K6DTUH64LRROQ
143,B00011JN5G,5.0,I love it,Great,AGEVTM6XYVZX454K6DTUH64LRROQ
144,B00011JN5G,5.0,I love it,Great,AGEVTM6XYVZX454K6DTUH64LRROQ
145,B00011JN5G,5.0,I love it,Great,AGEVTM6XYVZX454K6DTUH64LRROQ
146,B00011JN5G,5.0,I love it,Great,AGEVTM6XYVZX454K6DTUH64LRROQ


# Investigate the Duplicate Records

In [ ]:
#find reviews with the same product, user, and review text
duplicate_reviews = (
    reviews.groupby(["item_id", "user_id", "text"], dropna=False)
    .size()
    .reset_index(name="occurrences")
    .sort_values("occurrences", ascending=False)
)

duplicate_reviews.head(20)


,item_id,user_id,text,occurrences
1336,B07CKTWB2N,AHK3S32VWSXOBIYYF2MKGXUDRDBA,Evens out complexion.,10
3818,B0873Y1WFZ,AHJEGTHANR23UG3PMPA6MMCPV3RA,I wanted to get my daughter's hair braided for our trip. My stylist told me to get Kanekalon pre...,8
1334,B07CHB69Y3,AHUWZSKOOZ475MPMYUKUDQJQTYVQ,These bundles are so good I love them,8
1200,B074W1M23V,AHC4WQGKGVP5UQ2FRGWY6TWAOETA,"Lightweight, smells great, and oh does it softens the hair. Helps the most coarse hair to feel s...",8
1326,B07C6BDC2Q,AHWC5CGKFYDUDAVDUL2654BKBX6A,They go on well and stay on for several days. My friend recently bought another brand of metalli...,7
1297,B0798S8C41,AEFYI6LZU6TKJZ32LBOFO4CI2ZNA,Too expensive for such poor quality. There was no improvement and I am starting to think my scal...,7
3977,B087PY8ZCQ,AEZS34HKB5WZQHHJQ2NZQZ5PFZJA,"I have always had a problem of curly, frizzy, messy hair. And while I absolutely love my curls, ...",7
657,B014E2CSIQ,AGICMDY5MI55SBAEKWZNKMQX3TDQ,"Product was a bit dried out so thicker than usual. Same problem as other reviewer, it doesn’t pu...",7
707,B017O89LD2,AFVVSP4QRQCORIAEG4V4PIJQOXGA,One of the few products that have consistantly detangled my hair and I have 5 different textures.,7
438,B00Q1BNG6C,AG2MWE5FOYGZUXGKFDD2LBRWY4PQ,This item did nor work from the first day I received it,7


In [ ]:
print(
    "Review groups repeated more than once:",
    (duplicate_reviews["occurrences"] > 1).sum()
)

print(
    "Total rows belonging to repeated review groups:",
    duplicate_reviews.loc[
        duplicate_reviews["occurrences"] > 1,
        "occurrences"
    ].sum()
)
#I recommend treating this combination as the review's identity: item_id + user_id + text

Review groups repeated more than once: 893
Total rows belonging to repeated review groups: 2924


# Create the Cleaned Review Dataset

In [ ]:
reviews_clean = reviews.drop_duplicates(
    subset = ["item_id", "user_id", "text"]
).copy()
print("Original Review: ", len(reviews))
print("Cleaned Reviews: ", len(reviews_clean))
print("Reviews removed: ", len(reviews) - len(reviews_clean))

Original Review:  12219
Cleaned Reviews:  10188
Reviews removed:  2031


In [ ]:
print(
    "Remaining duplicate product-user-text combinations: ",
    reviews_clean.duplicated(
        subset = ['item_id', 'user_id', 'text']
    ).sum()
)

Remaining duplicate product-user-text combinations:  0


In [ ]:
#saving the new cleaned dataset as a csv
reviews_clean.to_csv("reviews_clean.csv", index=False)

# Verify the saved CSV

In [ ]:
#immediately load it
reviews_check = pd.read_csv("reviews_clean.csv")

print("Rows in cleaned DataFrame:", len(reviews_clean))
print("Rows in saved CSV:", len(reviews_check))
print("Columns match:", list(reviews_clean.columns) == list(reviews_check.columns))

Rows in cleaned DataFrame: 10188
Rows in saved CSV: 10188
Columns match: True


In [ ]:
#final check to make sure everything is working properly
print(reviews_check.head())
print("\nMissing values:")
print(reviews_check.isna().sum())

                        user_id     item_id  rating  \
0  AHCXZZ6R4US43PBVXXVN47OTTVHQ  B0002KHTCK     5.0   
1  AHUSGOTGVU5LFFPGCWLOTZ33RH7A  B0002KHTCK     5.0   
2  AFL2ZDLKD6WRY5AG35RAWWKQTJFA  B000PKKAGO     4.0   
3  AG2C3CTEG3T7HPEKIZH2OQ54722Q  B000PKKAGO     5.0   
4  AHHWI6X47RXVMZOZCKRZCGZG36CQ  B000PKNPES     3.0   

                                   title  \
0              Best Toothbrush Available   
1        Should have gotten this earlier   
2   Nice Shave...but beter than a razor?   
3  Made me a Convert (from Razor Blades)   
4  Norelco is great, this one is just OK   

                                                                                                  text  \
0  I have been using the Braun Oral B 3D toothbrush for some time now, and although that unit does ...   
1  Just used this thing this morning and oh boy...feel like I just get my teeth cleaned by the dent...   
2  At various times in my life I have rotated between using a razor vs a cordless shav

# Recheck Rating Distribution and Recheck Product Counts

In [ ]:
rating_counts_clean = reviews_clean["rating"].value_counts().sort_index()

print(rating_counts_clean)
print("\nPercentage of ratings:")
print(
    (reviews_clean["rating"].value_counts(normalize=True).sort_index() * 100)
    .round(2)
)

rating
1.0     332
2.0     389
3.0    1099
4.0    2539
5.0    5829
Name: count, dtype: int64

Percentage of ratings:
rating
1.0     3.26
2.0     3.82
3.0    10.79
4.0    24.92
5.0    57.21
Name: proportion, dtype: float64


In [ ]:
#recheck product review counts
review_counts_clean = (
    reviews_clean.groupby("item_id")
    .size()
    .sort_values(ascending=False)
)
print(review_counts_clean.describe())

count    2216.000000
mean        4.597473
std         3.674016
min         1.000000
25%         1.000000
50%         4.000000
75%         6.000000
max        47.000000
dtype: float64


In [ ]:
print("Products with at least 3 reviews:", (review_counts_clean >= 3).sum())
print("Products with at least 5 reviews:", (review_counts_clean >= 5).sum())
print("Products with at least 10 reviews:", (review_counts_clean >= 10).sum())
print("Products with at least 15 reviews:", (review_counts_clean >= 15).sum())
print("Products with at least 20 reviews:", (review_counts_clean >= 20).sum())

Products with at least 3 reviews: 1528
Products with at least 5 reviews: 900
Products with at least 10 reviews: 239
Products with at least 15 reviews: 27
Products with at least 20 reviews: 7


# Create Analysis Dataset

In [ ]:
product_analysis = (
    reviews_clean
    .groupby('item_id')
    .agg(
        review_count = ('rating','count'),
        avg_review_rating = ('rating', 'mean'),
        avg_helpful_votes = ('helpful_vote', 'mean')
    )
    .reset_index()
)

product_analysis = product_analysis[
    product_analysis['review_count'] >= 10
].copy()

print("Products in analysis: ", len(product_analysis))
print(product_analysis.head())

Products in analysis:  239
        item_id  review_count  avg_review_rating  avg_helpful_votes
120  B00KCTER3U            20           4.450000           0.300000
125  B00L4HJX7O            10           4.600000           0.700000
149  B00O2FGBJS            15           3.866667           0.066667
155  B00OS9YWJY            12           4.833333           0.916667
157  B00OXDWFG2            11           4.818182           0.363636


# Add Product Information

In [ ]:
product_analysis = product_analysis.merge(
    metadata[['item_id', 'title', 'main_category', 'store']],
    on = 'item_id',
    how = 'left'
)

print("Products in analysis: ", len(product_analysis))
print(product_analysis.head())

Products in analysis:  239
      item_id  review_count  avg_review_rating  avg_helpful_votes  \
0  B00KCTER3U            20           4.450000           0.300000   
1  B00L4HJX7O            10           4.600000           0.700000   
2  B00O2FGBJS            15           3.866667           0.066667   
3  B00OS9YWJY            12           4.833333           0.916667   
4  B00OXDWFG2            11           4.818182           0.363636   

                                                                                                 title  \
0  Retinol Face Moisturizer Cream Natural Facial moisturizing Cream 1.76 OZ with Ortho-Hydroxybenzo...   
1  OZNaturals Facial Mask - This Moisturizing Face Mask Contains Rosehip Oil, Pro Vitamin B5 & Amin...   
2  Argan Magic 100% Pure Argan Oil – Moisturizing Oil that Infuses Skin with Enriched Hydration | L...   
3  Pure Hyaluronic Acid Serum with Vitamin C for Face - Organic Anti Aging & Anti-Wrinkle Skin Trea...   
4                          N

# Create Sentiment Labels

In [ ]:
reviews_clean['sentiment'] = reviews_clean['rating'].apply(
    lambda x: "Positive" if x >= 4
    else "Neutral" if x == 3
    else "Negative"
)

print(reviews_clean['sentiment'].value_counts())
print()
print(reviews_clean['sentiment'].value_counts(normalize=True) * 100)

sentiment
Positive    8368
Neutral     1099
Negative     721
Name: count, dtype: int64

sentiment
Positive    82.135846
Neutral     10.787201
Negative     7.076953
Name: proportion, dtype: float64


# Adding Sentiment To The Product Level Analysis

In [ ]:
sentiment_summary = (
    reviews_clean
    .groupby('item_id')['sentiment']
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

#make sure all sentiment columns exist
for col in ["Positive", "Neutral", "Negative"]:
  if col not in sentiment_summary.columns:
    sentiment_summary[col] = 0

sentiment_summary = sentiment_summary.rename(columns={
    "Positive": "positive_reviews",
    "Neutral": "neutral_reviews",
    "Negative": "negative_reviews"
})

#calculate sentiment percentages
sentiment_summary['positive_pct'] = (
    sentiment_summary["positive_reviews"] /
    sentiment_summary[["positive_reviews", "neutral_reviews", "negative_reviews"]].sum(axis=1)
) * 100

sentiment_summary["negative_pct"] = (
    sentiment_summary["negative_reviews"] /
    sentiment_summary[["positive_reviews", "neutral_reviews", "negative_reviews"]].sum(axis=1)
) * 100

#add sentiment metrics to product analysis
product_analysis = product_analysis.merge(
    sentiment_summary,
    on="item_id",
    how="left"
)

print("Products in analysis: ", len(product_analysis))
print(product_analysis.head())

Products in analysis:  239
      item_id  review_count  avg_review_rating  avg_helpful_votes  \
0  B00KCTER3U            20           4.450000           0.300000   
1  B00L4HJX7O            10           4.600000           0.700000   
2  B00O2FGBJS            15           3.866667           0.066667   
3  B00OS9YWJY            12           4.833333           0.916667   
4  B00OXDWFG2            11           4.818182           0.363636   

                                                                                                 title  \
0  Retinol Face Moisturizer Cream Natural Facial moisturizing Cream 1.76 OZ with Ortho-Hydroxybenzo...   
1  OZNaturals Facial Mask - This Moisturizing Face Mask Contains Rosehip Oil, Pro Vitamin B5 & Amin...   
2  Argan Magic 100% Pure Argan Oil – Moisturizing Oil that Infuses Skin with Enriched Hydration | L...   
3  Pure Hyaluronic Acid Serum with Vitamin C for Face - Organic Anti Aging & Anti-Wrinkle Skin Trea...   
4                          N

# Create a Product Confidence Score
## **using a Bayesian style rated rating**

**R** = product's average rating

**v** = number of reviews for that product

**C** = overall average rating across the dataset

**m** = min review threshold

In [ ]:
#overview average rating across all cleaned reviews
C = reviews_clean["rating"].mean()

#minimum number of reviews required for product level analysis
m = 10

#calculated Bayesian weighted rating
product_analysis["weighted_rating"] = (
    (product_analysis["review_count"] /
     (product_analysis["review_count"] + m))
    * product_analysis["avg_review_rating"]
    +
    (m /
     (product_analysis["review_count"] + m))
    * C
)

print("Overall average rating:", round(C, 3))
print()
print(
    product_analysis[
        [
            "item_id",
            "review_count",
            "avg_review_rating",
            "positive_pct",
            "weighted_rating"
        ]
    ]
    .sort_values("weighted_rating", ascending=False)
    .head(10)
)

Overall average rating: 4.29

        item_id  review_count  avg_review_rating  positive_pct  \
137  B08HGZXLP6            23           4.869565    100.000000   
154  B08NJ5BTWG            13           4.923077    100.000000   
21   B07J3H27BJ            10           5.000000    100.000000   
178  B0998BD871            10           5.000000    100.000000   
222  B0B6QHG99T            12           4.916667    100.000000   
92   B088LWRYJC            17           4.823529     94.117647   
216  B0B5YVX9YY            11           4.909091    100.000000   
63   B085SY4WC3            13           4.846154    100.000000   
79   B087F14JSH            18           4.777778    100.000000   
208  B0B4KS62Z8            10           4.900000    100.000000   

     weighted_rating  
137         4.693983  
154         4.647889  
21          4.645073  
178         4.645073  
222         4.631884  
92          4.625980  
216         4.614355  
63          4.604411  
79          4.603623  
208         4

# Attach the product names to weighted rankings

In [ ]:
top_products = (
    product_analysis[
        [
            "item_id",
            "title",
            "store",
            "review_count",
            "avg_review_rating",
            "positive_pct",
            "negative_pct",
            "weighted_rating"
        ]
    ]
    .sort_values("weighted_rating", ascending=False)
    .head(10)
)

print(top_products.to_string(index=False))

   item_id                                                                                                                                                                                                    title       store  review_count  avg_review_rating  positive_pct  negative_pct  weighted_rating
B08HGZXLP6 Empty Amber Glass Spray Bottles - (4 Pack) 16 oz with Labels Refillable Container for Essential Oils, Cleaning Solutions, Cleaning Products, Hair, Plant Mister, Daily Shower, Gardening or Aromatherapy         AYL            23           4.869565    100.000000           0.0         4.693983
B08NJ5BTWG                   FASCY LAB GREEN+ Korean Sunscreen — SPF 50+PA++++, Daily Facial Sunscreen, Sun Cream With Patented Ingredients, Korean Spf, Korean Sun Cream for Face — Paraben Free [1 x 1.69 Fl.Oz.]       Fascy            13           4.923077    100.000000           0.0         4.647889
B07J3H27BJ                                                                                    